# Question 2 - MLP Comparison 


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import Modified_Code as orig
import numpy as np
import seaborn as sns
from sklearn.metrics
import confusion_matrix


In [ ]:
# Calling the MLP from the original script. Use this while in a different notebook
mlp_model = orig.InitialMLP().to(orig.device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

NameError: name 'orig' is not defined

In [ ]:
# Epoches output for trained MLP

# using MLP optimizer (use these lines in Colab)
# mlp_model = orig.InitialMLP().to(orig.device)
# optimizer = orig.optimizer

num_epochs = 15

mlp_train_losses = []
mlp_test_losses = []
mlp_train_accs = []
mlp_test_accs = []

print("Initiating the training the MLP model")

for epoch in range(num_epochs):
    train_loss, train_acc, _ = orig.train(mlp_model, orig.train_loader)
    
    test_loss, test_acc = orig.test(mlp_model, orig.test_loader)
    
    mlp_train_losses.append(train_loss)
    mlp_test_losses.append(test_loss)
    mlp_train_accs.append(train_acc)
    mlp_test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"MLP Train Loss: {train_loss:.4f} | MLP Train Accuracy: {train_acc:.2f}%")
    print(f"MLP Test Loss: {test_loss:.4f} | MLP Test Accuracy: {test_acc:.2f}%")
    print("-" * 50)

In [ ]:
# CNN and MLP weights vis.

# Extract the weights of the first layer in the MLP
mlp_weights = mlp_model.fc1.weight.detach().cpu()

# Set up the canvas to display 10 neurons (a 2x5 grid)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("MLP First Layer Weights Visualization (Reshaped to 32x32x3)", fontsize=14)

for i, ax in enumerate(axes.flat):
    # Extract the weight vector for the i-th neuron (length 3072)
    weight_vector = mlp_weights[i]
    
    # Reshape the vector back to a color image structure (C, H, W)
    img = weight_vector.view(3, 32, 32)
    
    # Normalize values to the 0-1 range for proper display
    img = (img - img.min()) / (img.max() - img.min())
    
    # Permute dimension order from (C, H, W) to (H, W, C) for matplotlib
    img = img.permute(1, 2, 0).numpy()
    
    ax.imshow(img)
    ax.set_title(f"Neuron {i+1}")
    ax.axis('off')

plt.show()

# Extract the weights of the first layer in the CNN
cnn_weights = orig.model.conv1.weight.detach().cpu()

# Set up the canvas (a 4x8 grid for the 32 filters)
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
fig.suptitle("CNN First Layer Filter Weights Visualization", fontsize=14)

for i, ax in enumerate(axes.flat):
    if i < cnn_weights.size(0):
        # Extract the i-th filter
        img = cnn_weights[i]
        
        # Normalize filter weight values to the 0-1 range for proper color display
        img = (img - img.min()) / (img.max() - img.min())
        
        # Permute dimension order from (C, H, W) to (H, W, C) for matplotlib
        img = img.permute(1, 2, 0).numpy()
        
        ax.imshow(img)
    ax.axis('off')

plt.show()

In [ ]:
# Learning curves 

import matplotlib.pyplot as plt



# Set up a canvas with 1 row and 2 columns for side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
epochs = range(1, 16)  # 15 epochs

# -------------------------------------------------------------
# Graph 7: Loss Comparison (CNN vs MLP)
# -------------------------------------------------------------
# CNN curves
ax1.plot(epochs, train_losses, label='CNN Train Loss', color='#1f77b4', linestyle='-', linewidth=2)
ax1.plot(epochs, test_losses, label='CNN Test Loss', color='#ff7f0e', linestyle='-', linewidth=2)

# MLP curves
ax1.plot(epochs, mlp_train_losses, label='MLP Train Loss', color='#1f77b4', linestyle='--', linewidth=2)
ax1.plot(epochs, mlp_test_losses, label='MLP Test Loss', color='#ff7f0e', linestyle='--', linewidth=2)

ax1.set_title('Graph 7: Train & Test Loss Comparison (CNN vs MLP)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.set_xticks(epochs)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(fontsize=10, loc='upper right')

# -------------------------------------------------------------
# Graph 8: Accuracy Comparison (CNN vs MLP)
# -------------------------------------------------------------
# CNN curves
ax2.plot(epochs, train_accs, label='CNN Train Accuracy', color='#2ca02c', linestyle='-', linewidth=2)
ax2.plot(epochs, test_accs, label='CNN Test Accuracy', color='#d62728', linestyle='-', linewidth=2)

# MLP curves
ax2.plot(epochs, mlp_train_accs, label='MLP Train Accuracy', color='#2ca02c', linestyle='--', linewidth=2)
ax2.plot(epochs, mlp_test_accs, label='MLP Test Accuracy', color='#d62728', linestyle='--', linewidth=2)

ax2.set_title('Graph 8: Train & Test Accuracy Comparison (CNN vs MLP)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Accuracy (%)', fontsize=11)
ax2.set_xticks(epochs)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(fontsize=10, loc='lower right')

# Adjust layout and display the visualization
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix for evaluation (CNN VS MLP)

# Confusion matrix MLP 


def plot_mlp_confusion_matrix(mlp_model, loader, device, title="MLP Confusion Matrix"):
    """
    Evaluates the trained MLP model on the test dataset and plots a 
    normalized confusion matrix to analyze classification errors.
    """
    # Set the model to evaluation mode (disables dropout/batchnorm updates)
    mlp_model.eval()
    
    all_preds = []
    all_labels = []
    
    # Disable gradient computation for faster inference and memory efficiency
    with torch.no_grad():
        for images, labels in loader:
            # Move data to the active hardware accelerator (GPU or CPU)
            images = images.to(device)
            
            # Forward pass: compute predicted outputs by passing images to the model
            outputs = mlp_model(images)
            
            # Get the index of the highest log-probability (predicted class)
            _, preds = outputs.max(1)
            
            # Collect predictions and true labels back to CPU as numpy arrays
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Official target class names ordered by CIFAR-10 indexing
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                   'dog', 'frog', 'horse', 'ship', 'truck']
    
    # Generate the raw confusion matrix mapping actual vs. predicted labels
    raw_cm = confusion_matrix(all_labels, all_preds)
    
    # Normalize the matrix rows to display percentages instead of raw sample counts
    normalized_cm = raw_cm.astype('float') / raw_cm.sum(axis=1)[:, np.newaxis] * 100

    # Initialize a clean canvas for heat map plotting
    plt.figure(figsize=(10, 8))
    
    # Render the confusion matrix using seaborn's annotated heatmap
    sns.heatmap(normalized_cm, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    
    # Formatting titles and axes labels for the academic report
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    
    # Optimize layout geometry and present the final figure
    plt.tight_layout()
    plt.show()

# Execute the visualization using your trained MLP model
# We label this as Graph 9 to be presented right before the CNN matrix
plot_mlp_confusion_matrix(
    mlp_model,           # Your trained MLP instance from the current notebook
    orig.test_loader,    # The evaluation DataLoader pulled from the .py file
    orig.device,         # Target device (CUDA or CPU)
    title="Confusion Matrix 4: MLP Confusion Matrix"
)


# Confusion matrix CNN

def plot_cnn_confusion_matrix(cnn_model, loader, device, title="CNN Confusion Matrix"):
    """
    Evaluates the trained CNN model on the test dataset and plots a 
    normalized confusion matrix to analyze classification errors.
    """
    # Set the model to evaluation mode (disables dropout/batchnorm updates)
    cnn_model.eval()
    
    all_preds = []
    all_labels = []
    
    # Disable gradient computation for faster inference and memory efficiency
    with torch.no_grad():
        for images, labels in loader:
            # Move data to the active hardware accelerator (GPU or CPU)
            images = images.to(device)
            
            # Forward pass: compute predicted outputs by passing images to the model
            outputs = cnn_model(images)
            
            # Get the index of the highest log-probability (predicted class)
            _, preds = outputs.max(1)
            
            # Collect predictions and true labels back to CPU as numpy arrays
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Official target class names ordered by CIFAR-10 indexing
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                   'dog', 'frog', 'horse', 'ship', 'truck']
    
    # Generate the raw confusion matrix mapping actual vs. predicted labels
    raw_cm = confusion_matrix(all_labels, all_preds)
    
    # Normalize the matrix rows to display percentages instead of raw sample counts
    normalized_cm = raw_cm.astype('float') / raw_cm.sum(axis=1)[:, np.newaxis] * 100

    # Initialize a clean canvas for heat map plotting
    plt.figure(figsize=(10, 8))
    
    # Render the confusion matrix using seaborn's annotated heatmap
    sns.heatmap(normalized_cm, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    
    # Formatting titles and axes labels for the academic report
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    
    # Optimize layout geometry and present the final figure
    plt.tight_layout()
    plt.show()

# Execute the visualization using the trained CNN model pulled from python script
# We label this as Graph 10 assuming  MLP was Graph 9
plot_cnn_confusion_matrix(
    orig.model,          # The trained CNN instance from the .py script
    orig.test_loader,    # The evaluation DataLoader
    orig.device,         # Target device (CUDA or CPU)
    title="Confusion Matrix 3: CNN Confusion Matrix"
)